# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### The rule, in plain words

A page is worth reviewing if it gets enough search exposure to judge, and earns a smaller
share of the clicks its ranking position normally delivers.

"Normally delivers" is measured, not assumed: each position tier's expected CTR is the
weighted CTR that tier actually achieves in this data, and a page's score is the shortfall
as a **share** of that expectation — not as a count of clicks.

The rule ignores staleness entirely, because `days_since_last_update` holds about 20 batch
values across 30,000 rows and cannot support a freshness claim.

Reason code: `underclicking_for_position`. Action: `review_title_and_meta` when it fires,
`no_action` when the page meets expectation, `insufficient_exposure` when there was not
enough traffic to judge.

### Signal A — better position earns higher CTR → CONFIRMED

Flag-linked: this is the signal behind FlyRank's CTR-fix logic. Weighted CTR per bucket
(total clicks ÷ total impressions), not the mean of per-row rates — per-row `ctr` has a
median of 0.07 and a max of 100.00 from a single impression.

| position_tier | n | weighted CTR % |
|---|---|---|
| top_3 | 1,116 | 0.489 |
| page_1 | 11,814 | 0.350 |
| striking | 7,304 | 0.347 |
| page_3_5 | 7,242 | 0.155 |
| deep | 1,319 | 0.041 |

Monotonic, every bucket over the 50-row floor. Two caveats. `page_1` and `striking` differ
by 0.003 points across 19,000 rows, so this is four usable levels, not five. And
`avg_position > 0` is filtered first: 1,205 unranked pages are mislabeled into `top_3`
upstream, which is 52% of that bucket.

This table is not just a check — it becomes the rule. Each tier's weighted CTR is the
expectation every page is scored against.

### Signal B — a traffic floor makes CTR stable enough to act on → MIXED

Flag-linked: volume is the signal behind the quick-win flag. I set the floor at 250
impressions in 90 days and tested whether CTR settles above it. If volume stabilises CTR,
the p90–p10 spread should shrink as impressions rise.

| impressions bucket | n | median CTR | p90 − p10 |
|---|---|---|---|
| 1-99 | 6,789 | 0.00 | 1.75 |
| 100-249 | 2,620 | 0.00 | 0.88 |
| 250-999 | 5,874 | 0.00 | 0.55 |
| 1000-4999 | 7,361 | 0.16 | 0.57 |
| 5000+ | 6,151 | 0.22 | 0.66 |

It tightens to 0.55, then widens again to 0.57 and 0.66. Not monotonic, so MIXED.

The median column is the more damaging finding. CTR's median is 0.00 in the `250-999`
bucket — more than half the pages just above my chosen floor earned no clicks at all. And
zero-click share tracks exposure, not quality: 51.4% at 250-999, 14.6% at 1000-4999, 1.3%
at 5000+, and by tier 10.9% at page_1 rising to 74.6% at deep.

**This check changed the rule.** A page in the `250-999` bucket ranking page_1 expects
about 0.9 clicks. Zero is the most likely single outcome, so scoring it as a 100% shortfall
would rank noise at the top. An impressions floor cannot fix that, because the number of
impressions needed to earn one expected click depends entirely on the tier — a deep page
needs roughly 12,200.

So the floor is set in **expected clicks**, not impressions: `impressions_90d ×
tier_expected_ctr ≥ 5`. At that floor the zero-click share falls to 3.8% and the rule covers
37.4% of ranked pages. `deep` is out of scope — only 12 rows clear the floor, below the
50-row threshold, so no verdict is reported for that tier.

In [40]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

pd.set_option("display.width", 120)

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)
print("Working dir:", os.getcwd())
print("Loaded:", df.shape)

os.makedirs("work/outputs", exist_ok=True)

# --- Signal A (flag-linked: the CTR-fix logic) ---
# Claim: better average position earns higher CTR.
# Guard on avg_position > 0 -- position_tier's top_3 bucket is 52% unranked pages.
ranked = df[df["avg_position"] > 0].copy()
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

sig_a = (
    ranked.groupby("position_tier")
    .agg(n=("ctr", "count"),
         total_impressions=("impressions_90d", "sum"),
         total_clicks=("clicks_90d", "sum"))
    .reindex(pos_order)
)
sig_a["expected_ctr_pct"] = sig_a["total_clicks"] / sig_a["total_impressions"] * 100
print("--- Signal A: position tier vs. weighted CTR (best -> worst) ---")
print(sig_a[["n", "expected_ctr_pct"]].round(3).to_string())

# --- Signal B (flag-linked: the quick-win volume flag) ---
# Claim: below a traffic floor, CTR is too noisy to act on.
# If true, the spread of per-page CTR should shrink as impressions rise.
bins = [0, 99, 249, 999, 4999, np.inf]
labels = ["1-99", "100-249", "250-999", "1000-4999", "5000+"]
ranked["imp_bucket"] = pd.cut(ranked["impressions_90d"], bins=bins, labels=labels)

sig_b = (
    ranked.groupby("imp_bucket", observed=True)["ctr"]
    .agg(n="count",
         median_ctr="median",
         p10=lambda s: s.quantile(0.10),
         p90=lambda s: s.quantile(0.90))
)
sig_b["p90_minus_p10"] = sig_b["p90"] - sig_b["p10"]
print("\n--- Signal B: CTR spread by impression volume ---")
print(sig_b.round(3).to_string())

kept = (ranked["impressions_90d"] >= 250).sum()
print(f"\nrows at or above the 250-impression floor: {kept} of {len(ranked)} ranked "
      f"({kept / len(ranked) * 100:.1f}%)")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded: (30000, 44)
--- Signal A: position tier vs. weighted CTR (best -> worst) ---
                   n  expected_ctr_pct
position_tier                         
top_3           1116             0.489
page_1         11814             0.350
striking        7304             0.347
page_3_5        7242             0.155
deep            1319             0.041

--- Signal B: CTR spread by impression volume ---
               n  median_ctr   p10   p90  p90_minus_p10
imp_bucket                                             
1-99        6789        0.00  0.00  1.75           1.75
100-249     2620        0.00  0.00  0.88           0.88
250-999     5874        0.00  0.00  0.55           0.55
1000-4999   7361        0.16  0.00  0.57           0.57
5000+       6151        0.22  0.05  0.71           0.66

rows at or above the 250-i

In [41]:
above = ranked[ranked["impressions_90d"] >= 250].copy()
above["zero_click"] = above["clicks_90d"] == 0

print(f"--- rows >= 250 impressions: {len(above)} ---")
print(f"zero-click share: {above['zero_click'].mean() * 100:.1f}%")

print("\n--- zero-click share by position tier ---")
print(
    above.groupby("position_tier")
    .agg(n=("zero_click", "size"), zero_click_pct=("zero_click", lambda s: round(s.mean() * 100, 1)))
    .reindex(pos_order)
    .to_string()
)

print("\n--- zero-click share by impression bucket ---")
print(
    above.groupby("imp_bucket", observed=True)
    .agg(n=("zero_click", "size"), zero_click_pct=("zero_click", lambda s: round(s.mean() * 100, 1)))
    .to_string()
)

--- rows >= 250 impressions: 19386 ---
zero-click share: 21.5%

--- zero-click share by position tier ---
                  n  zero_click_pct
position_tier                      
top_3           496            16.9
page_1         7811            10.9
striking       5234            21.1
page_3_5       5224            31.8
deep            621            74.6

--- zero-click share by impression bucket ---
               n  zero_click_pct
imp_bucket                      
250-999     5874            51.4
1000-4999   7361            14.6
5000+       6151             1.3


In [42]:
tier_ctr = sig_a["expected_ctr_pct"] / 100

scored = ranked.copy()
scored["tier_expected_ctr"] = scored["position_tier"].map(tier_ctr)
scored["expected_clicks"] = scored["impressions_90d"] * scored["tier_expected_ctr"]

for floor in [1, 3, 5, 10]:
    keep = scored["expected_clicks"] >= floor
    zc = (scored.loc[keep, "clicks_90d"] == 0).mean() * 100
    print(f"expected_clicks >= {floor:2d}  ->  {keep.sum():6d} rows "
          f"({keep.mean() * 100:4.1f}% of ranked), zero-click share {zc:4.1f}%")

print("\n--- tier mix at expected_clicks >= 5 ---")
print(
    scored[scored["expected_clicks"] >= 5]
    .groupby("position_tier").size().reindex(pos_order).to_string()
)

expected_clicks >=  1  ->   17324 rows (60.2% of ranked), zero-click share 15.8%
expected_clicks >=  3  ->   12967 rows (45.0% of ranked), zero-click share  6.6%
expected_clicks >=  5  ->   10757 rows (37.4% of ranked), zero-click share  3.8%
expected_clicks >= 10  ->    7744 rows (26.9% of ranked), zero-click share  1.4%

--- tier mix at expected_clicks >= 5 ---
position_tier
top_3        391
page_1      5603
striking    2892
page_3_5    1859
deep          12


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [43]:
import json

TIER_CTR = sig_a["expected_ctr_pct"] / 100
EXPECTED_CLICKS_FLOOR = 5.0
IN_SCOPE_TIERS = ["page_1", "striking", "page_3_5", "top_3"]  # deep excluded: 12 rows clear the floor

q = df[df["avg_position"] > 0].copy()
q = q.reset_index(drop=True)
q["tier_expected_ctr"] = q["position_tier"].map(TIER_CTR)
q["expected_clicks"] = q["impressions_90d"] * q["tier_expected_ctr"]
q["missed_clicks"] = q["expected_clicks"] - q["clicks_90d"]

scorable = (
    (q["expected_clicks"] >= EXPECTED_CLICKS_FLOOR)
    & q["position_tier"].isin(IN_SCOPE_TIERS)
)
fires = scorable & (q["missed_clicks"] > 0)

# FROZEN BASELINE: the shortfall as a SHARE of expected clicks, not a count.
# The absolute-count version is kept below for the record -- it ranked worse than
# random because missed_clicks scales with impressions, so it sorted the highest-traffic
# (and most stable) pages to the top. See section 4.
q["baseline_score"] = np.where(fires, q["missed_clicks"] / q["expected_clicks"], 0.0)
q["score_absolute_deprecated"] = np.where(fires, q["missed_clicks"], 0.0)
q["reason_code"] = np.where(fires, "underclicking_for_position", "")
q["action"] = np.select(
    [fires, scorable & ~fires],
    ["review_title_and_meta", "no_action"],
    default="insufficient_exposure",
)
order = q.sort_values(["baseline_score", "expected_clicks"], ascending=[False, False]).index
q["baseline_rank"] = pd.Series(range(1, len(order) + 1), index=order)
q["rank_absolute_deprecated"] = q["score_absolute_deprecated"].rank(method="first", ascending=False).astype(int)

cols = [
    "content_id", "client_id", "position_tier", "impressions_90d", "clicks_90d", "ctr",
    "tier_expected_ctr", "expected_clicks", "missed_clicks",
    "baseline_score", "reason_code", "action", "baseline_rank",
    "score_absolute_deprecated", "rank_absolute_deprecated",
]
queue = q.sort_values("baseline_rank")[cols]
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"wrote work/outputs/baseline_action_score.csv  ({len(queue)} rows)")

print("\n--- action breakdown ---")
print(q["action"].value_counts().to_frame("n")
      .assign(pct=lambda d: (d["n"] / len(q) * 100).round(1)).to_string())

# --- precision@K. trend_direction is the LABEL and is not an input anywhere above. ---
def labels_by(rank_col):
    return (q.sort_values(rank_col)["trend_direction"].str.lower() == "down").astype(int).values

lab_frozen = labels_by("baseline_rank")
lab_deprecated = labels_by("rank_absolute_deprecated")
base_rate = float(lab_frozen.mean())
KS = [10, 50, 100, 500]

print(f"\n--- precision@K (base rate {base_rate:.3f}) ---")
print(f"{'K':>5} {'frozen (ratio)':>16} {'lift':>8} {'deprecated (abs)':>18} {'lift':>8}")
for k in KS:
    pf, pd_ = float(lab_frozen[:k].mean()), float(lab_deprecated[:k].mean())
    print(f"{k:>5} {pf:>16.3f} {pf - base_rate:>+8.3f} {pd_:>18.3f} {pd_ - base_rate:>+8.3f}")

metrics = {
    "rule": "underclicking_for_position",
    "frozen_score": "missed_clicks / expected_clicks (share of expectation)",
    "deprecated_score": "missed_clicks (absolute count) -- ranked below base rate at every K",
    "expected_clicks_floor": EXPECTED_CLICKS_FLOOR,
    "in_scope_tiers": IN_SCOPE_TIERS,
    "n_ranked": int(len(q)),
    "n_fired": int(fires.sum()),
    "base_rate": round(base_rate, 4),
    "flag_lift": {
        "fired_declining_rate": round(float(
            q.loc[fires, "trend_direction"].str.lower().eq("down").mean()), 4),
        "no_action_declining_rate": round(float(
            q.loc[scorable & ~fires, "trend_direction"].str.lower().eq("down").mean()), 4),
    },
    "precision_at_k_frozen": {str(k): round(float(lab_frozen[:k].mean()), 4) for k in KS},
    "precision_at_k_deprecated": {str(k): round(float(lab_deprecated[:k].mean()), 4) for k in KS},
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nwrote work/outputs/baseline_metrics.json")

ties = int((q["baseline_score"] == 1.0).sum())
fired = int((q["reason_code"] != "").sum())
print(f"rows tied at exactly 1.0: {ties}  ({ties / fired * 100:.1f}% of fired)")
print(f"\nexpected_clicks among tied rows:")
print(q.loc[q["baseline_score"] == 1.0, "expected_clicks"]
      .describe(percentiles=[.5, .9, .99]).round(1).to_string())

wrote work/outputs/baseline_action_score.csv  (28795 rows)

--- action breakdown ---
                           n   pct
action                            
insufficient_exposure  18050  62.7
review_title_and_meta   7176  24.9
no_action               3569  12.4

--- precision@K (base rate 0.564) ---
    K   frozen (ratio)     lift   deprecated (abs)     lift
   10            0.600   +0.036              0.400   -0.164
   50            0.800   +0.236              0.500   -0.064
  100            0.810   +0.246              0.490   -0.074
  500            0.774   +0.210              0.512   -0.052

wrote work/outputs/baseline_metrics.json
rows tied at exactly 1.0: 405  (5.6% of fired)

expected_clicks among tied rows:
count    405.0
mean      11.6
std       36.6
min        5.0
50%        7.6
90%       17.0
99%       52.8
max      731.0


In [44]:
r = q["baseline_rank"]
print("rank is a clean 1..n permutation:", r.is_unique and r.min() == 1 and r.max() == len(q))
print("n rows:", len(q), " n unique ranks:", r.nunique())

chk = q.sort_values("baseline_rank")["baseline_score"]
print("score is non-increasing along rank:", bool((chk.diff().dropna() <= 1e-12).all()))
print("\nscore at ranks 1, 405, 406, 500, 501:")
print(q.set_index("baseline_rank")["baseline_score"].reindex([1, 405, 406, 500, 501]).round(4).to_string())

rank is a clean 1..n permutation: True
n rows: 28795  n unique ranks: 28795
score is non-increasing along rank: True

score at ranks 1, 405, 406, 500, 501:
baseline_rank
1      1.0000
405    1.0000
406    0.9873
500    0.9387
501    0.9382


In [45]:
top = q.sort_values("baseline_rank").head(500)
print("--- top 500 by baseline_score: traffic profile ---")
print(f"median impressions_90d, top 500 : {top['impressions_90d'].median():,.0f}")
print(f"median impressions_90d, all ranked: {q['impressions_90d'].median():,.0f}")

fired = q[q["reason_code"] == "underclicking_for_position"].copy()
fired["decile"] = pd.qcut(fired["baseline_score"], 10, labels=False, duplicates="drop")
d = (fired.assign(declining=fired["trend_direction"].str.lower().eq("down"))
     .groupby("decile")
     .agg(n=("declining", "size"),
          median_impressions=("impressions_90d", "median"),
          declining_rate=("declining", "mean")))
d["declining_rate"] = (d["declining_rate"] * 100).round(1)
print("\n--- declining rate by score decile (9 = highest score) ---")
print(d.round(0).to_string())

print("\n--- declining rate by action label ---")
print(q.assign(declining=q["trend_direction"].str.lower().eq("down"))
      .groupby("action").agg(n=("declining", "size"),
                             declining_rate=("declining", "mean")).round(3).to_string())

--- top 500 by baseline_score: traffic profile ---
median impressions_90d, top 500 : 3,090
median impressions_90d, all ranked: 828

--- declining rate by score decile (9 = highest score) ---
          n  median_impressions  declining_rate
decile                                         
0       718              6810.0            54.0
1       718              6293.0            59.0
2       717              6966.0            59.0
3       718              5960.0            58.0
4       717              6176.0            59.0
5       718              5532.0            64.0
6       717              5307.0            64.0
7       718              5960.0            68.0
8       717              4002.0            71.0
9       718              3891.0            76.0

--- declining rate by action label ---
                           n  declining_rate
action                                      
insufficient_exposure  18050           0.549
no_action               3569           0.504
review_title_

## 3. Top-10 review

The queue's top ten, read as a skeptic. All ten fire `underclicking_for_position`, all ten
score the capped 1.0, and all ten earned zero clicks — so the same objection applies to
every row, and it is not a small one.

| # | tier | impressions | expected clicks | declining? |
|---|---|---|---|---|
| 1 | page_1 | 208,678 | 731.0 | yes |
| 2 | striking | 17,622 | 61.1 | yes |
| 3 | page_1 | 16,786 | 58.8 | yes |
| 4 | page_1 | 16,156 | 56.6 | yes |
| 5 | page_1 | 15,101 | 52.9 | no |
| 6 | page_1 | 14,519 | 50.9 | no |
| 7 | page_1 | 13,676 | 47.9 | yes |
| 8 | striking | 12,275 | 42.6 | no |
| 9 | striking | 8,779 | 30.5 | no |
| 10 | page_1 | 7,737 | 27.1 | yes |

**Action for all ten: `review_title_and_meta`.** The rule's logic is that these pages
already rank — page one or striking distance — so the body content is doing its job and
the click is being lost at the search result. Title and description are where that gets
fixed.

**Why they are here.** Each page's expected clicks is its impressions times the weighted
CTR its position tier actually achieves in this data. Every one of these earned none of
that expectation, so the shortfall is 100% and the score saturates.

**What would make it wrong — the same thing for all ten, and it is structural.** Rank 1
has 208,678 impressions on page one and zero clicks. Expected 731. A page ranking that
well across that much exposure does not earn zero clicks from real searchers. The far more
likely explanation is that clicks are not being recorded for these rows. Ranks 2 through
10 have the same shape at 27 to 61 expected clicks — high enough that a genuine zero has
no plausible chance of occurring naturally.

So the top of this queue is a data-quality queue, not a content queue. A content lead sent
to rewrite these titles would spend the week on pages whose measurement is broken, and the
titles would test as unchanged because the clicks were never being counted.

**Two smaller notes.** Ranks 5, 6, 8, and 9 are labelled not-declining, so four of ten
disagree with the label even before the tracking objection — P@10 is 0.600, the weakest
precision figure in the notebook (P@100 is 0.810). And these ten are not ordered by
strength of finding: all 405 rows scoring exactly 1.0 are tied, and the tie-break sorts by
exposure, which puts the largest pages first. Rank 1 is the biggest tied page, not the
best pick.

**What I did not do.** I did not filter these rows out. Dropping them would raise P@10 and
hide the finding. The score is behaving correctly; the input is not. The fix belongs
upstream — a `clicks_tracked` guard before scoring — not as a threshold tweak here.

In [46]:
top10 = q.sort_values("baseline_rank").head(10).copy()
top10["declining"] = top10["trend_direction"].str.lower().eq("down")

show = [
    "baseline_rank", "position_tier", "impressions_90d", "clicks_90d", "ctr",
    "expected_clicks", "missed_clicks", "baseline_score", "action", "declining",
]
print("--- top 10 by baseline_score (share of expected clicks missed) ---")
print(top10[show].round(3).to_string(index=False))

print("\n--- how close to the floor is the head of the queue? ---")
print(f"top 10  median expected_clicks: {top10['expected_clicks'].median():.1f}")
print(f"top 500 median expected_clicks: {q.sort_values('baseline_rank').head(500)['expected_clicks'].median():.1f}")
print(f"floor: {EXPECTED_CLICKS_FLOOR}")

print("\n--- zero-click share along the queue ---")
ranked_q = q.sort_values("baseline_rank")
for k in [10, 50, 100, 500]:
    head = ranked_q.head(k)
    print(f"top {k:>3}: zero-click {(head['clicks_90d'] == 0).mean() * 100:>5.1f}%   "
          f"declining {head['trend_direction'].str.lower().eq('down').mean() * 100:>5.1f}%")

ties = (q["baseline_score"] == 1.0).sum()
print(f"rows tied at score exactly 1.0: {ties}")
print(f"share of fired rows: {ties / (q['reason_code'] != '').sum() * 100:.1f}%")
print(f"\nexpected_clicks among the tied rows:")
print(q.loc[q["baseline_score"] == 1.0, "expected_clicks"].describe(percentiles=[.5, .9, .99]).round(1).to_string())

--- top 10 by baseline_score (share of expected clicks missed) ---
 baseline_rank position_tier  impressions_90d  clicks_90d  ctr  expected_clicks  missed_clicks  baseline_score                action  declining
             1        page_1           208678           0  0.0          731.049        731.049             1.0 review_title_and_meta       True
             2      striking            17622           0  0.0           61.127         61.127             1.0 review_title_and_meta       True
             3        page_1            16786           0  0.0           58.805         58.805             1.0 review_title_and_meta       True
             4        page_1            16156           0  0.0           56.598         56.598             1.0 review_title_and_meta       True
             5        page_1            15101           0  0.0           52.902         52.902             1.0 review_title_and_meta      False
             6        page_1            14519           0  0.0       

## 4. Weak picks + leakage check

### Weak picks

**The head of the queue is the weakest part of it.** Ranks 1–405 all score exactly 1.0.
The score is the shortfall as a share of expected clicks, so any page with zero clicks
saturates and the score cannot separate them. That is 5.6% of fired rows, and it swallows
K=10, K=50, and K=100 entirely — those three precision figures measure my tie-break, not my
score. P@500 is the first number that reaches past the tie.

**The tie-break costs precision.** Within the tied block I order by expected clicks
descending: among pages earning zero, more exposure is stronger evidence. That is
defensible to someone who has never seen the label, so I kept it. It also fronts the
highest-traffic pages, and high-traffic pages are more stable — which is why P@10 is 0.600
against P@100 at 0.810. Ordering the ties the other way would have scored better. I did
not do that, because choosing a tie-break by which precision it produces is fitting to the
label.

**The top ten are probably tracking failures, not content failures.** 25 rows across the
dataset have zero clicks while expecting 20 or more; ten of them are my top ten. Rank 1
expects 731 clicks from 208,678 page-one impressions and reports zero. 32% of those 25 rows
belong to one client — the same client that was 81.8% missing on `word_count` in
`w04_signal_audit.ipynb`. n=25 is below any sample floor, so I read this as consistent with
the earlier finding rather than as independent proof of it. The `word_count` evidence
(7,008 rows) is what carries the weight.

**The score I shipped is the second one I wrote.** The first ranked by absolute missed
clicks and came in *below* the base rate at every K (0.400 / 0.500 / 0.490 / 0.512 against
0.564). Missed clicks scales with impressions, so it sorted the largest and most stable
pages to the top — declining rate ran 60% in the bottom score decile down to 54% in the
top. Switching to the shortfall as a share of expectation reversed it: 54% up to 76% across
deciles, and top-500 median impressions fell from 50,358 to 3,090. Both scores stay in
`baseline_action_score.csv` as `baseline_score` and `score_absolute_deprecated` so the
comparison is checkable.

**What holds regardless of any of the above.** Firing the reason code at all is worth +6.9
points: pages flagged `underclicking_for_position` decline at 63.3% versus 50.4% for
`no_action`. That number depends on set membership, not ordering, so no tie-break or
ranking choice touches it. It is the most defensible figure here.

### Leakage check

**Label columns are never inputs.** `is_declining_label` derives from `trend_direction`,
which derives from `trend_pct`. All three are excluded from the score, the floor, the tier
expectations, and the reason code. `trend_direction` appears only where precision@K and the
declining rates are computed, after ranking is complete.

**No 30-day windows.** `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`,
`clicks_prev_30d`, `sessions_last_30d`, and `sessions_prev_30d` appear nowhere. Earlier
work in `w03_feature_leakage_check.ipynb` showed the label reconstructs from those columns.

**No product flags.** The score uses no FlyRank reason code as an input. `page_one_decay_risk`
in particular is absent — `w04_signal_audit.ipynb` found it inverted across eight slices.

**One leak I cannot remove, stated plainly.** `impressions_90d` and `clicks_90d` are
trailing-90-day windows that overlap the period the label describes. Every usable traffic
column on this CSV has that property, so a rule baseline cannot avoid it. It applies
equally to the Week-5 model, which will read the same columns, so the baseline-versus-model
comparison stays fair — but neither number should be read as a forecast. This is measured,
directional, decision-support, on one dataset.

**Coverage limit.** The rule scores 37.4% of ranked pages. `deep` is out of scope: at
0.041% expected CTR a deep page needs roughly 12,200 impressions to clear a five-click
floor, and only 12 rows qualify — below the 50-row floor, so I report no verdict for that
tier rather than a number built on 12 rows.

In [47]:
susp = q[(q["clicks_90d"] == 0) & (q["expected_clicks"] >= 20)]
print(f"zero-click pages expecting >= 20 clicks: {len(susp)}")
print(f"  ...of which declining: {susp['trend_direction'].str.lower().eq('down').mean() * 100:.1f}%")
print(f"  impressions_90d median: {susp['impressions_90d'].median():,.0f}")
print(f"\nzero-click share of all ranked rows: {(q['clicks_90d'] == 0).mean() * 100:.1f}%")
print("\n--- do these cluster in one client? ---")
print(susp["client_id"].value_counts().head(5).to_frame("n")
      .assign(pct=lambda d: (d["n"] / len(susp) * 100).round(1)).to_string())

zero-click pages expecting >= 20 clicks: 25
  ...of which declining: 72.0%
  impressions_90d median: 7,732

zero-click share of all ranked rows: 41.7%

--- do these cluster in one client? ---
                   n   pct
client_id                 
client_19581e27de  8  32.0
client_4e07408562  3  12.0
client_3fdba35f04  3  12.0
client_6208ef0f77  2   8.0
client_7f2253d7e2  2   8.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.